In [ ]:
# --- bootstrap: this notebook lives inside the module folder (graph_partitioning/),
# --- so hop to the repo root before importing, so `import graph_partitioning` and the
# --- repo-root-relative data paths (results/..., graph_partitioning/tests/...) resolve.
import os, sys, subprocess
_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("repo root:", _root)


# Design-Space Exploration (Synthetic Pool)

Parameter sweeps, cluster sizing, and capacity planning using a synthetic device
pool. None of this depends on real RPi measurements.

**Launch from the repo root with the venv active:**
```bash
source graph_partitioning/.venv/bin/activate
jupyter notebook design_exploration.ipynb
```

**Sections**
1. Setup & imports
2. Comprehensive parameter sweeps
3. Find minimum cluster size
4. Voice assistant workload sizing
5. Model workload capacity
6. Device pool CDFs
7. Inference memory timeline

*Hardware vs simulator validation lives in `hw_calibration.ipynb`.*


## 1. Setup

In [ ]:
%matplotlib inline

from __future__ import annotations
import csv, datetime, os, sys

import matplotlib.pyplot as plt
import numpy as np

import graph_partitioning.graph_plots as graph_plots
from graph_partitioning import Device, ModelSpec, Network, Simulator
from graph_partitioning.common import CommunicationModel, PeerToPeerPolicy
from graph_partitioning.graph_plots import set_save_figures
from graph_partitioning.logging_utils import setup_output_dir
from graph_partitioning.sweep import (
    POLICY_COLOR, POLICY_RUNNERS,
    default_servers, run_all_policies_quiet, run_all_sweeps,
)
from graph_partitioning.analysis_plots import (
    plot_capacity_preset,
    plot_memory_timeline, plot_latency_timeline, plot_best_policy_timeline,
    sample_points as make_sample_points,
    TTFT_BUDGETS,
)

OUTPUT_ROOT = "sim_output"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# --- Figure-saving gate -------------------------------------------------------
# Set False while iterating to skip writing PDFs (figures still display inline).
# This one flag gates BOTH the direct savefig() calls in the cells below AND the
# library plot helpers (run_all_sweeps, plot_capacity_preset, plot_*_timeline),
# which check the same graph_plots.SAVE_FIGURES flag. CSVs are always written.
# Re-run this cell after changing SAVE_FIGS.
SAVE_FIGS = True
set_save_figures(SAVE_FIGS)


def savefig(path, *, fig=None, **kw):
    """Save the current (or given) figure only when SAVE_FIGS is on."""
    if graph_plots.SAVE_FIGURES:
        (fig or plt).savefig(path, bbox_inches="tight", **kw)


def human_time(s):
    """Compact human-readable duration for plot legends (µs/ms/s/min/h)."""
    if s is None or s != s or s == float("inf"):   # None / NaN / inf
        return "—"
    if s < 1e-3:
        return f"{s * 1e6:.0f} µs"
    if s < 1:
        return f"{s * 1e3:.0f} ms"
    if s < 60:
        return f"{s:.3g} s"
    if s < 3600:
        m, sec = divmod(round(s), 60)
        return f"{m:d} min {sec:02d} s"
    h, rem = divmod(round(s), 3600)
    return f"{h:d} h {rem // 60:02d} min"


print(f"Ready. SAVE_FIGS={SAVE_FIGS}")

In [ ]:
# num_layers = transformer blocks only (TinyLlama-1.1B has 22). The output head
# offloaded by `-ngl 23` is a single [d_model->vocab] projection, not a block;
# it's accounted separately via embedding_weights_gb(), so it must NOT inflate
# num_layers.
TINYLLAMA = ModelSpec(
    name="TinyLlama-1.1B",
    num_layers=22, num_heads=32, num_kv_heads=4,
    d_model=2048, d_k=64, d_ff=5632, vocab_size=32000,
)

LLAMA_7B = ModelSpec(
    name="Llama-7B",
    num_layers=32, num_heads=32, num_kv_heads=32,
    d_model=4096, d_k=128, d_ff=11008, vocab_size=32000,
)

LLAMA_13B = ModelSpec(
    name="Llama-13B",
    num_layers=40, num_heads=40, num_kv_heads=40,
    d_model=5120, d_k=128, d_ff=13824, vocab_size=32000,
)

MODEL_PRESETS = [("tinyllama", TINYLLAMA), ("llama-7b", LLAMA_7B), ("llama-13b", LLAMA_13B)]

## 4. Comprehensive Parameter Sweeps

Sweeps device count, model size, bandwidth, seq_len, and gen_tokens using the
**synthetic** device pool (so we can scale to large N). Outputs PDFs and CSVs.

Expected runtime: ~5–15 min depending on sweep resolution.

In [ ]:
sweep_out = run_all_sweeps(
    device_counts=(2, 3, 4, 6, 8),
    # Model-size axis uses the real presets from §2 (no synthetic scale factors);
    # add entries to MODEL_PRESETS for larger models. The model sweep auto-sizes
    # its cluster so the largest preset fits.
    models=[m for _, m in MODEL_PRESETS],
    bandwidths_mbps=(10, 50, 100, 200, 500),
    seq_lens=(1, 16, 64, 256),
    gen_tokens_list=(1, 8, 32, 128),
)
print(f"Sweep outputs: {sweep_out}")

## 5. Find Minimum Cluster Size

For each model preset and device count N, runs every policy and finds the smallest
N that hits usable TTFT and TPT targets.

Uses the synthetic device pool (cycles through 10-device specs).

In [ ]:
CONTEXT_LEN      = 256
GEN_TOKENS       = 64
DEVICE_COUNTS    = [2, 4, 6, 8, 12, 16, 24, 32]
TTFT_THRESHOLD_S = 1.0   # interactive chatbot bar
TPT_THRESHOLD_S  = 0.1   # 10 tok/s comfortable streaming


def _build_net_synthetic(servers):
    c = Device("coordinator", gflops=100.0, memory_gb=16.0)
    n = Network(c, servers, default_bw_mbps=200.0, default_rtt_ms=18.0)
    n.set_comm_config(CommunicationModel.PEER_TO_PEER,
                      peer2peer_policy=PeerToPeerPolicy.ALL_TO_ALL)
    return n


def run_cluster_sizing(model, n_devices):
    svrs = default_servers(n_devices)
    net_a = _build_net_synthetic(svrs)
    ttft  = run_all_policies_quiet(model, net_a, CONTEXT_LEN, 1,
                                   kv_cache_seq_len=CONTEXT_LEN)
    net_b = _build_net_synthetic(svrs)
    mid   = CONTEXT_LEN + GEN_TOKENS // 2
    tpt   = run_all_policies_quiet(model, net_b, 1, 1, kv_cache_seq_len=mid)
    return {
        p: {"ttft_s": float(ttft.get(p, float("nan"))),
            "tpt_s":  float(tpt.get(p,  float("nan")))}
        for p in POLICY_RUNNERS
    }

In [ ]:
cluster_results = {}
for preset_name, model in MODEL_PRESETS:
    cluster_results[preset_name] = {}
    for n in DEVICE_COUNTS:
        cluster_results[preset_name][n] = run_cluster_sizing(model, n)
    print(f"Done: {preset_name}")

print("Cluster sizing complete.")

In [ ]:
fig, axes = plt.subplots(len(MODEL_PRESETS), 2, figsize=(14, 4 * len(MODEL_PRESETS)))

for row, (preset_name, model) in enumerate(MODEL_PRESETS):
    data = cluster_results[preset_name]
    ax_ttft, ax_tpt = axes[row]

    for policy in POLICY_RUNNERS:
        xs = DEVICE_COUNTS
        ttfts = [data[n][policy]["ttft_s"] for n in xs]
        tpts  = [data[n][policy]["tpt_s"]  for n in xs]
        color = POLICY_COLOR.get(policy, "gray")
        ax_ttft.plot(xs, ttfts, marker="o", label=policy, color=color)
        ax_tpt.plot(xs,  tpts,  marker="s", label=policy, color=color)

    # Lowest (best-achievable) TTFT / TPT across all policies & device counts.
    best = lambda key: min((data[n][p][key] for p in POLICY_RUNNERS for n in DEVICE_COUNTS
                            if data[n][p][key] < float("inf")), default=None)
    best_ttft, best_tpt = best("ttft_s"), best("tpt_s")
    if best_ttft is not None:
        ax_ttft.axhline(best_ttft, color="green", linestyle=":", linewidth=1.2,
                        label=f"min TTFT {human_time(best_ttft)}")
    if best_tpt is not None:
        ax_tpt.axhline(best_tpt, color="green", linestyle=":", linewidth=1.2,
                       label=f"min TPT {human_time(best_tpt)}")

    ax_ttft.axhline(TTFT_THRESHOLD_S, color="red", linestyle="--", linewidth=1,
                    label=f"target {human_time(TTFT_THRESHOLD_S)}")
    ax_tpt.axhline(TPT_THRESHOLD_S,   color="red", linestyle="--", linewidth=1,
                   label=f"target {human_time(TPT_THRESHOLD_S)}")

    # log y: latencies span orders of magnitude across policies (e.g. single_node
    # swapping vs alpa), so a log scale keeps every policy legible.
    ax_ttft.set(title=f"{preset_name} — TTFT", xlabel="# devices",
                ylabel="TTFT (s)", yscale="log")
    ax_tpt.set( title=f"{preset_name} — TPT",  xlabel="# devices",
                ylabel="TPT (s/tok)", yscale="log")
    ax_ttft.legend(fontsize=7)
    ax_tpt.legend(fontsize=7)

plt.tight_layout()
savefig(f"{OUTPUT_ROOT}/cluster_sizing.pdf")
plt.show()

## 6. Voice Assistant Workload Sizing

Workload: long private-context prompt (~2048 tokens), short spoken response (~50 tokens).

Targets:
- TTFT ≤ 5 s (user tolerates a "consulting" pause)
- TPT ≤ 0.25 s/tok (≥ 4 tok/s matches natural TTS speech rate)

In [ ]:
VA_PROMPT_TOKENS  = 2048
VA_RESPONSE_TOKENS = 50
VA_TTFT_TARGET_S  = 5.0
VA_TPT_TARGET_S   = 0.25
VA_DEVICE_COUNTS  = [2, 4, 6, 8, 12, 16, 24, 32, 48, 64]


def run_va_sizing(model, n_devices):
    svrs = default_servers(n_devices)
    net_a = _build_net_synthetic(svrs)
    ttft  = run_all_policies_quiet(model, net_a, VA_PROMPT_TOKENS, 1,
                                   kv_cache_seq_len=VA_PROMPT_TOKENS)
    net_b = _build_net_synthetic(svrs)
    mid   = VA_PROMPT_TOKENS + VA_RESPONSE_TOKENS // 2
    tpt   = run_all_policies_quiet(model, net_b, 1, 1, kv_cache_seq_len=mid)
    return {
        p: {"ttft_s": float(ttft.get(p, float("nan"))),
            "tpt_s":  float(tpt.get(p,  float("nan")))}
        for p in POLICY_RUNNERS
    }

In [ ]:
va_results = {}
for preset_name, model in MODEL_PRESETS:
    va_results[preset_name] = {}
    for n in VA_DEVICE_COUNTS:
        va_results[preset_name][n] = run_va_sizing(model, n)
    print(f"Done: {preset_name}")

In [ ]:
fig, axes = plt.subplots(len(MODEL_PRESETS), 2, figsize=(14, 4 * len(MODEL_PRESETS)))

for row, (preset_name, model) in enumerate(MODEL_PRESETS):
    data = va_results[preset_name]
    ax_ttft, ax_tpt = axes[row]

    for policy in POLICY_RUNNERS:
        xs    = VA_DEVICE_COUNTS
        ttfts = [data[n][policy]["ttft_s"] for n in xs]
        tpts  = [data[n][policy]["tpt_s"]  for n in xs]
        color = POLICY_COLOR.get(policy, "gray")
        ax_ttft.plot(xs, ttfts, marker="o", label=policy, color=color)
        ax_tpt.plot(xs,  tpts,  marker="s", label=policy, color=color)

    # Lowest (best-achievable) TTFT / TPT across all policies & device counts.
    best = lambda key: min((data[n][p][key] for p in POLICY_RUNNERS for n in VA_DEVICE_COUNTS
                            if data[n][p][key] < float("inf")), default=None)
    best_ttft, best_tpt = best("ttft_s"), best("tpt_s")
    if best_ttft is not None:
        ax_ttft.axhline(best_ttft, color="green", linestyle=":", linewidth=1.2,
                        label=f"min TTFT {human_time(best_ttft)}")
    if best_tpt is not None:
        ax_tpt.axhline(best_tpt, color="green", linestyle=":", linewidth=1.2,
                       label=f"min TPT {human_time(best_tpt)}")

    ax_ttft.axhline(VA_TTFT_TARGET_S, color="red", linestyle="--", linewidth=1,
                    label=f"target {human_time(VA_TTFT_TARGET_S)}")
    ax_tpt.axhline(VA_TPT_TARGET_S,   color="red", linestyle="--", linewidth=1,
                   label=f"target {human_time(VA_TPT_TARGET_S)}")
    # log y: latencies span orders of magnitude across policies.
    ax_ttft.set(title=f"{preset_name} — TTFT (voice)", xlabel="# devices",
                ylabel="TTFT (s)", yscale="log")
    ax_tpt.set( title=f"{preset_name} — TPT (voice)",  xlabel="# devices",
                ylabel="TPT (s/tok)", yscale="log")
    ax_ttft.legend(fontsize=7)
    ax_tpt.legend(fontsize=7)

plt.tight_layout()
savefig(f"{OUTPUT_ROOT}/voice_assistant_sizing.pdf")
plt.show()

## 7. Model Workload Capacity

For each model preset, finds the largest prompt that fits within TTFT budgets
and the steady-state TPT the cluster can deliver.

*(Corresponds to `model_workload_capacity.py`)*

In [ ]:
from graph_partitioning.smart_home_devices import smart_home_servers

CAPACITY_PRESETS = [
    ("tinyllama", TINYLLAMA, "small"),   # 4 small smart-home devices
    ("llama-7b",  LLAMA_7B,  "medium"),  # 32 medium devices
    ("llama-13b", LLAMA_13B, "large"),   # 48 large devices
]
PROMPT_LENGTHS  = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
SPEECH_BUDGET_S = 10.0


def best_ttft_tpt(model, preset, prompt_len):
    """Return (best_ttft_s, best_ttft_policy, best_tpt_s, best_tpt_policy)."""
    def _build(svrs):
        c = Device("coordinator", gflops=100.0, memory_gb=16.0)
        n = Network(c, svrs, default_bw_mbps=200.0, default_rtt_ms=18.0)
        n.set_comm_config(CommunicationModel.PEER_TO_PEER,
                          peer2peer_policy=PeerToPeerPolicy.ALL_TO_ALL)
        return n
    def _best(d):
        # (value, policy) so the smallest latency sorts first and unpacks as
        # value-then-policy.
        valid = [(v, p) for p, v in d.items() if not np.isnan(v)]
        return min(valid, key=lambda vp: vp[0]) if valid else (float("nan"), "")

    ttft = run_all_policies_quiet(model, _build(smart_home_servers(preset)),
                                  prompt_len, 1, kv_cache_seq_len=prompt_len)
    tpt  = run_all_policies_quiet(model, _build(smart_home_servers(preset)),
                                  1, 1, kv_cache_seq_len=prompt_len + 25)
    tt_s, tt_p = _best(ttft)
    tp_s, tp_p = _best(tpt)
    return tt_s, tt_p, tp_s, tp_p


out_dir = setup_output_dir(f"{OUTPUT_ROOT}/model_workload_capacity")
os.makedirs(out_dir, exist_ok=True)

for tag, model, preset in CAPACITY_PRESETS:
    servers = smart_home_servers(preset)
    n = len(servers)
    print(f"\n=== {model.name} — {preset} cluster (N={n}) ===")
    rows = []
    for prompt in PROMPT_LENGTHS:
        tt, tt_p, tp, tp_p = best_ttft_tpt(model, preset, prompt)
        rows.append({"prompt_len": prompt, "ttft_s": tt, "ttft_policy": tt_p,
                     "tpt_s": tp, "tpt_policy": tp_p})
        print(f"  prompt={prompt:>5d}  TTFT={tt:.2f}s ({tt_p})  TPT={tp:.4f}s ({tp_p})")

    for budget in TTFT_BUDGETS:
        fit = [r for r in rows if not np.isnan(r["ttft_s"]) and r["ttft_s"] <= budget]
        if not fit:
            print(f"  → ≤{budget}s : no prompt fits")
            continue
        best = max(fit, key=lambda r: r["prompt_len"])
        max_resp = int((SPEECH_BUDGET_S - best["ttft_s"]) / best["tpt_s"]) if best["tpt_s"] > 0 else 0
        print(f"  → ≤{budget}s : max prompt {best['prompt_len']} tok, "
              f"TPT {best['tpt_s']:.3f} s/tok → {max_resp} tok in {SPEECH_BUDGET_S:.0f}s")

    plot_capacity_preset(
        model.name, preset, n, rows,
        save_path=os.path.join(out_dir, f"capacity_{tag}.pdf"),
    )
    plt.show()

print(f"\nPlots saved to {out_dir}/")

## 8. Device Pool CDFs

Distributions of compute, DRAM, and swap bandwidth across the synthetic 10-device
pool used in sections 5–7.

In [ ]:
from graph_partitioning.smart_home_devices import list_preset_names, preset_summary, smart_home_servers

POOL_SIZE = 10
pool = default_servers(POOL_SIZE)
gflops = [d.gflops            for d in pool]
dram   = [d.memory_gb          for d in pool]
swap   = [d.swap_bandwidth_mbps for d in pool]


def _stairs(values):
    arr = np.sort(np.array(values, dtype=float))
    n = len(arr)
    xs = np.concatenate(([arr[0]], np.repeat(arr, 2), [arr[-1] * 1.05]))
    ys = np.concatenate(([0.0], np.repeat(np.arange(1, n + 1) / n, 2)))
    return xs[:2 * n + 1], ys[:2 * n + 1]


def _plot_cdf(ax, values, *, unit, color, title):
    xs, ys = _stairs(values)
    ax.plot(xs, ys, color=color, linewidth=2.5)
    ax.scatter(np.sort(values), np.arange(1, len(values) + 1) / len(values),
               color=color, s=40, zorder=5)
    median = float(np.median(values))
    ax.axvline(median, color="black", linestyle=":", linewidth=1, alpha=0.6)
    ax.text(median, 0.04, f" median={median:g} {unit}", fontsize=8)
    ax.set(xlabel=f"({unit})", ylabel="CDF", title=title)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
_plot_cdf(axes[0], gflops, unit="GFLOPS", color="#264b7b", title="Compute")
_plot_cdf(axes[1], dram,   unit="GB",     color="#71b171", title="DRAM")
_plot_cdf(axes[2], swap,   unit="Mbps",   color="#e38f45", title="Swap BW")
fig.suptitle(f"Synthetic device pool CDFs ({POOL_SIZE}-device pool)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(f"{OUTPUT_ROOT}/device_pool_cdf.pdf")
plt.show()

## 9. Inference Memory Timeline

Per-device memory usage as the KV cache grows through prefill and decode.
Dashed lines mark each device's DRAM limit — crossing into swap is visible.

*(Corresponds to `plot_inference_timeline.py`)*

In [ ]:
from graph_partitioning.sweep import run_all_policies_with_memory

# --- Config (adjust as needed) ---
TIMELINE_CONTEXT  = 512
TIMELINE_GEN      = 128
TIMELINE_SAMPLES  = 24
TIMELINE_MODEL    = TINYLLAMA   # swap for LLAMA_7B etc.
TIMELINE_N_DEVS   = 4

# --- Build network ---
svrs    = default_servers(TIMELINE_N_DEVS)
net_tl  = _build_net_synthetic(svrs)

pts = make_sample_points(TIMELINE_CONTEXT, TIMELINE_GEN, TIMELINE_SAMPLES)
print(f"Sampling {len(pts)} KV-cache lengths: {pts[:5]} … {pts[-3:]}")

# --- Run all policies at each sample point ---
timeline = []
for kv_len in pts:
    net_kv = _build_net_synthetic(default_servers(TIMELINE_N_DEVS))
    timeline.append(run_all_policies_with_memory(
        TIMELINE_MODEL, net_kv, 1, 1, kv_cache_seq_len=kv_len,
    ))
print("Done.")

# --- Plot ---
out_dir = setup_output_dir(f"{OUTPUT_ROOT}/inference_timeline")
os.makedirs(out_dir, exist_ok=True)

dev_names = [d.name for d in svrs]
dev_dram  = [d.memory_gb for d in svrs]
label     = TIMELINE_MODEL.name

plot_memory_timeline(
    pts, timeline, dev_names, dev_dram, TIMELINE_CONTEXT,
    save_path=os.path.join(out_dir, "memory_timeline.pdf"),
    title=f"Per-device memory — {label}",
)
plt.show()

plot_latency_timeline(
    pts, timeline, TIMELINE_CONTEXT,
    save_path=os.path.join(out_dir, "latency_timeline.pdf"),
    title=f"Per-decode-step latency — {label}",
)
plt.show()

plot_best_policy_timeline(
    pts, timeline, TIMELINE_CONTEXT,
    save_path=os.path.join(out_dir, "best_policy_timeline.pdf"),
    title=f"Best policy along timeline — {label}",
)
plt.show()

print(f"Plots saved to {out_dir}/")

## 10. Alpa Intra-Op ILP — network/memory-adaptive sharding

The `alpa` policy implements Alpa's **intra-operator** pass as a real ILP (CBC/HiGHS
via PuLP): for every operator it picks a sharding strategy — `replicate`
(full weights+compute on each device, no collective) or `shard`
(tensor-parallel, 1/k compute + an all-reduce) — to minimise
`Σ(compute + intra-op collectives) + Σ(resharding)`, subject to the device
mesh and a memory-feasibility constraint (replicate is only legal if the whole
model fits one device).

The result is **adaptive**, which is the point of an Alpa baseline:

- **Slow network** → collectives are expensive → the ILP **replicates** (no comm).
- **Fast network** → collectives are cheap → it **shards** for the compute speedup.
- **Model too big for one device** → replicate is infeasible → it is **forced to shard**
  and must pay the all-reduce cost.

*(Alpa's inter-operator/pipeline DP is a throughput optimisation that amortises
the pipeline bubble across microbatches; single-request decode is B=1, so that
pass degenerates and is omitted.)*

In [ ]:
import pandas as pd
from graph_partitioning.policies.alpa import simulate_alpa, _SOLVER

print(f"ILP solver: {type(_SOLVER).__name__}")


def _alpa_net(bw_mbps, rtt_ms, mem_gb, n=4):
    """A small CLIENT_SERVER mesh of n identical RPi-class devices."""
    coordinator = Device("coordinator", gflops=100.0, memory_gb=16.0)
    devs = [Device(f"rpi{i}", gflops=2.5, memory_gb=mem_gb, swap_bandwidth_mbps=240.0)
            for i in range(1, n + 1)]
    net = Network(coordinator, devs, default_bw_mbps=bw_mbps, default_rtt_ms=rtt_ms)
    net.set_comm_config(CommunicationModel.CLIENT_SERVER)
    return net


# Three regimes that should drive the ILP to different sharding decisions.
scenarios = [
    ("slow Wi-Fi\n50 Mbps · 20 ms",        _alpa_net(50,      20.0, 2.0)),
    ("fast fabric\n100 Gbps · 0.05 ms",    _alpa_net(100_000, 0.05, 2.0)),
    ("slow + model too big\nfor 1 dev (0.2 GB)", _alpa_net(50, 20.0, 0.2)),
]

rows = []
for label, net in scenarios:
    r = simulate_alpa(TINYLLAMA, net, seq_len=1, gen_tokens=1)
    sp = r["selected_plan"]
    counts = sp["strategy_counts"]
    rows.append({
        "scenario": label.replace("\n", " — "),
        "replicate_ops": counts.get("replicate", 0),
        "shard_ops": counts.get("shard", 0),
        "intra_op_s": round(sp["intra_op_cost_s"], 3),
        "total_s": round(r["total_latency_s"], 3),
        "plan_ms": round(r["planning_time_s"] * 1000, 1),
    })
display(pd.DataFrame(rows))

# Stacked bar: how many operators the ILP replicated vs sharded in each regime.
fig, ax = plt.subplots(figsize=(8.5, 4.6))
labels = [s[0] for s in scenarios]
rep = [r["replicate_ops"] for r in rows]
shd = [r["shard_ops"] for r in rows]
total_ops = rep[0] + shd[0]
ax.bar(labels, rep, label="replicate (no collective)", color="#264b7b")
ax.bar(labels, shd, bottom=rep, label="shard (tensor-parallel + all-reduce)", color="#e38f45")
for i, r in enumerate(rows):
    ax.text(i, total_ops + 1.5, f"{r['total_s']} s/tok", ha="center",
            fontsize=9, fontweight="bold")
ax.set(ylabel="# operators", ylim=(0, total_ops * 1.12),
       title="Alpa intra-op ILP — sharding chosen per regime (TinyLlama, 4 devices)")
ax.legend(loc="upper center")
plt.tight_layout()
savefig(f"{OUTPUT_ROOT}/alpa_ilp_sharding.pdf")
plt.show()

### 10.1 Planning cost per policy

Every policy reports `planning_time_s` — the wall-clock to **compute the
allocation** (device selection + partitioning + cost model, **excluding
visualization**), which matters for an adaptive system that re-plans online.
Analytical/greedy policies (`single_node`, `pipeline`, `hybrid_pp_tp`, `tensor`)
are ~microseconds; METIS graph partitioning is ~1 ms; the **optimization-based
baselines are the heaviest** — the Alpa intra-op **ILP** and the m-SCT
favorite-child **LP** each cost a few-to-tens of milliseconds, the price of
solving a real math program over the operator graph. (METIS's inline diagnostic
plotting is timed separately and subtracted, so this is a fair comparison.)

In [ ]:
from graph_partitioning.policies.single_node import simulate_single_node
from graph_partitioning.policies.pipeline import simulate_pipeline
from graph_partitioning.policies.tensor import simulate_tensor_parallel
from graph_partitioning.policies.hybrid_pp_tp import simulate_hybrid_pp_tp
from graph_partitioning.policies.msct import simulate_msct

runners = [
    ("single_node", simulate_single_node),
    ("pipeline", simulate_pipeline),
    ("tensor", simulate_tensor_parallel),
    ("hybrid_pp_tp", simulate_hybrid_pp_tp),
    ("alpa", simulate_alpa),
    ("msct", simulate_msct),
]
try:
    from graph_partitioning.policies.metis import simulate_metis
    runners.append(("metis", simulate_metis))
except Exception as exc:  # pymetis is an optional dependency
    print(f"(metis skipped: {exc})")

# Average a few runs so the sub-millisecond policies are measurable.
REPS = 5
prows = []
for name, fn in runners:
    times = []
    for _ in range(REPS):
        res = fn(TINYLLAMA, _alpa_net(50, 20.0, 2.0), seq_len=1, gen_tokens=1)
        times.append(res.get("planning_time_s", float("nan")) * 1000)
    prows.append({"policy": name, "planning_ms": round(np.mean(times), 3)})

pdf = pd.DataFrame(prows).sort_values("planning_ms").reset_index(drop=True)
display(pdf)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.barh(pdf["policy"], pdf["planning_ms"].clip(lower=1e-3), color="#71b171")
ax.set(xscale="log", xlabel="planning time (ms, log scale)",
       title="Assignment/allocation compute time per policy (TinyLlama, 4 devices)")
for y, v in enumerate(pdf["planning_ms"]):
    ax.text(max(v, 1e-3), y, f"  {v:g} ms", va="center", fontsize=9)
plt.tight_layout()
savefig(f"{OUTPUT_ROOT}/planning_time_per_policy.pdf")
plt.show()

## 11. Why Alpa's inter-op pipeline DP doesn't help single-request decode

Alpa's full algorithm puts an **inter-operator** pass on top of the intra-op
ILP: a DP that slices the model into pipeline **stages** on device **submeshes**
and amortises the pipeline **bubble** across `B` microbatches. The GPipe makespan
is `Σ stage_time + (B−1)·max stage_time` — so the bubble (idle pipeline slots)
only pays off when many microbatches flow through.

`alpa_pipeline_search` implements that inter-op search (per stage count: split
the cluster into contiguous submeshes, cost each stage's sharding with the same
intra-op ILP, evaluate the GPipe makespan). Sweeping `B` shows the negative
result for **single-request decode**:

- **B = 1** (one token in flight) → the makespan-optimal plan is **1 stage** —
  pipelining adds serial hops with no bubble to fill.
- the optimal depth grows **only** as `B` grows (i.e. when *serving many
  concurrent requests*).
- at small `B` the bubble fraction is huge — most pipeline slots sit idle.

That's why the `alpa` policy omits the inter-op DP: it's a *throughput*
optimisation, and single-request autoregressive decode has no concurrency to
exploit.

In [ ]:
from graph_partitioning.policies.alpa import alpa_pipeline_search

# Wireless edge cluster of 8 RPi-class devices (reuses _alpa_net from §10).
pnet = _alpa_net(50, 20.0, 2.0, n=8)

# 1) Optimal pipeline depth vs concurrency B.
Bs = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
opt_stages = [alpa_pipeline_search(TINYLLAMA, pnet, seq_len=1, microbatches=B)["optimal_stages"]
              for B in Bs]

# 2) At B=1, per-request latency and bubble vs stage count.
r1 = alpa_pipeline_search(TINYLLAMA, pnet, seq_len=1, microbatches=1)
stages   = [x["stages"] for x in r1["results"]]
lat_b1   = [x["per_request_latency_s"] for x in r1["results"]]

display(pd.DataFrame({"B": Bs, "optimal_#stages": opt_stages}))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

ax1.plot(Bs, opt_stages, marker="o", color="#264b7b")
ax1.axhline(1, ls=":", color="gray")
ax1.set(xscale="log", xlabel="microbatches B  (concurrent requests)",
        ylabel="optimal # pipeline stages",
        title="Inter-op DP optimum vs concurrency")
ax1.annotate("single-request decode (B=1)\n→ 1 stage (no pipeline)",
             xy=(1, 1), xytext=(3, max(opt_stages) * 0.6),
             arrowprops=dict(arrowstyle="->"), fontsize=9)

ax2.plot(stages, lat_b1, marker="s", color="#be6c6c")
ax2.set(xlabel="# pipeline stages", ylabel="per-request latency (s/tok)",
        title="At B=1, pipelining only ADDS latency\n(each stage = another serial hop)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
display(fig)
plt.close(fig)

print(f"B=1 optimum: {opt_stages[0]} stage(s)  |  "
      f"B={Bs[-1]} optimum: {opt_stages[-1]} stages  "
      f"(pipelining only pays off with concurrency)")